In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=8e14b3962ce368caed4f893b217320b3efa1ea30d436755c3780e132003aecb8
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [3]:
# Quantum random number generation and qubit encode/measure.
# These helpers are used by all agents in the protocol.

backend = BasicSimulator()

def quantum_random_bit():
    # Generate a truly random bit by preparing the superposition state |+> = 1/sqrt(2) * (|0> + |1>) via a Hadamard gate on |0>, then measuring.
    # The outcome is 0 or 1 with equal probability 1/2.

    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    # Generate a list of n quantum random bits.
    return [quantum_random_bit() for _ in range(n)]


def encode_qubit(bit, basis):
    # Encode a classical bit into a qubit using the given basis.
    # Returns a QuantumCircuit representing the prepared qubit.
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc


def measure_qubit(qc, basis):
    # Measure a qubit circuit in the given basis.
    # Returns the measured bit (0 or 1).
    qc_m = qc.copy()
    if basis == 1:
        qc_m.h(0)
    qc_m.measure_all()
    compiled = transpile(qc_m, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [4]:
# Total number of qubits to send
N = 100

# Alice generates her secret random bit values
alice_bits = quantum_random_bits(N)

# Alice generates her random basis choices
alice_bases = quantum_random_bits(N)

# Alice encodes each bit into a qubit and sends it to Bob
qubits_sent = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print("ALICE")
print(f"Bits sent (first 20): {alice_bits[:20]}")
print(f"Bases chosen (first 20): {alice_bases[:20]}")
print(f"Alice has encoded and sent {N} qubits.")

ALICE
Bits sent (first 20): [1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0]
Bases chosen (first 20): [1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0]
Alice has encoded and sent 100 qubits.


In [5]:
# Bob generates his random basis choices, independent of Alice's
bob_bases = quantum_random_bits(N)

# Bob measures each received qubit in his chosen basis
bob_results = [measure_qubit(qubits_sent[i], bob_bases[i]) for i in range(N)]

print("BOB")
print(f"Bases chosen (first 20): {bob_bases[:20]}")
print(f"Results (first 20): {bob_results[:20]}")
print(f"Bob has measured all {N} qubits.")

BOB
Bases chosen (first 20): [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1]
Results (first 20): [1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0]
Bob has measured all 100 qubits.


In [6]:
# Sifting
sifted_alice = []
sifted_bob = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:   # Bases match: keep this bit
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print("SIFTING")
print(f"Total qubits sent: {N}")
print(f"Bits retained after sift: {len(sifted_alice)} ({100*len(sifted_alice)/N:.1f}%)")
print(f"Sifted Alice (first 20): {sifted_alice[:20]}")
print(f"Sifted Bob (first 20): {sifted_bob[:20]}")

SIFTING
Total qubits sent: 100
Bits retained after sift: 46 (46.0%)
Sifted Alice (first 20): [1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]
Sifted Bob (first 20): [1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]


In [7]:
# Error Checking
SAMPLE_SIZE = max(1, len(sifted_alice) // 5)  # Sacrifice ~20% for error checking
THRESHOLD = 0.05  # Abort if error rate > 5%

sample_alice = sifted_alice[:SAMPLE_SIZE]
sample_bob = sifted_bob[:SAMPLE_SIZE]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / SAMPLE_SIZE

print("ERROR CHECKING")
print(f"Check sample size: {SAMPLE_SIZE} bits")
print(f"Errors found: {errors}")
print(f"Error rate: {error_rate:.1%}")
print()

if error_rate > THRESHOLD:
    print("High error rate.")
    final_alice_key = []
    final_bob_key = []
else:
    print("Error rate within bounds.")
    # Remaining sifted bits (after removing the check sample) form the key
    final_alice_key = sifted_alice[SAMPLE_SIZE:]
    final_bob_key = sifted_bob[SAMPLE_SIZE:]

print()
print("FINAL KEY")
print(f"Key length: {len(final_alice_key)} bits")
print(f"Alice's key: {final_alice_key}")
print(f"Bob's key: {final_bob_key}")

ERROR CHECKING
Check sample size: 9 bits
Errors found: 0
Error rate: 0.0%

Error rate within bounds.

FINAL KEY
Key length: 37 bits
Alice's key: [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0]
Bob's key: [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0]


In [8]:
# Verification
if final_alice_key and final_bob_key:
    mismatches = sum(a != b for a, b in zip(final_alice_key, final_bob_key))
    print("VERIFICATION")
    if mismatches == 0:
        print("Keys are matching.")
        print(f"BB84 (no attacker) succeeded: {len(final_alice_key)}-bit shared key established.")
    else:
        print(f"Keys differ in {mismatches} position(s) — unexpected error.")
else:
    print("Key exchange was aborted — no shared key to verify.")

VERIFICATION
Keys are matching.
BB84 (no attacker) succeeded: 37-bit shared key established.
